# Real-World Patterns & Troubleshooting

## What's covered

- A worked **end-to-end AWS example** — VPC + ALB + ASG + RDS — built as composed modules
- **Organizing very large state** — splitting by blast radius, the rule of thumb for one-state-per-component
- **Drift detection** in production — what to monitor, how to alert
- **Cost control** — tagging discipline, Infracost, common money-burning patterns
- The **debugging workflow** when Terraform misbehaves — `TF_LOG`, state surgery, the common error patterns
- The honest question — **when Terraform is the wrong tool**
- The **series throughline** in one paragraph


## The end-to-end example — a small web app

Everything in the series, applied. A small web application on AWS:

- A **VPC** with three public and three private subnets across three availability zones.
- An **Application Load Balancer** in the public subnets.
- An **Auto Scaling Group** of EC2 instances in the private subnets, behind the ALB, running the web app.
- An **RDS Postgres** database in the private subnets, accessible only from the ASG.
- **IAM roles** wired correctly: the ASG instances can read from S3 and write to CloudWatch.
- **CloudWatch alarms** on high CPU, high error rate, low free disk.
- A **CloudFront distribution** in front of the ALB for caching and TLS termination.

That's the shape. The implementation lives across four composed modules:

```
   live/prod/
   ├── main.tf
   ├── terraform.tfvars
   └── backend.tf

   modules/
   ├── network/    <- VPC, subnets, IGW, NAT, route tables
   ├── compute/    <- ASG, launch template, security groups
   ├── data/       <- RDS, parameter group, subnet group
   └── edge/       <- ALB, target group, CloudFront, ACM cert
```

Each module is responsible for one part of the stack. Dependencies flow upward: edge depends on compute (target group attaches to ASG), compute depends on data (instances have RDS connection string), all depend on network.


## The root config

The root `live/prod/main.tf` is a thin orchestration layer that wires the modules together:

```hcl
terraform {
  required_version = ">= 1.6"
  required_providers {
    aws = { source = "hashicorp/aws", version = "~> 5.0" }
  }
}

provider "aws" {
  region = var.region
}

data "aws_caller_identity" "current" {}

locals {
  env = "prod"
  common_tags = {
    Environment = local.env
    Service     = "webapp"
    ManagedBy   = "terraform"
    Owner       = "platform-team"
  }
}

module "network" {
  source = "../../modules/network"

  name = "${local.env}-webapp"
  cidr = var.vpc_cidr
  azs  = var.azs
  tags = local.common_tags
}

module "data" {
  source = "../../modules/data"

  name                = "${local.env}-webapp"
  vpc_id              = module.network.vpc_id
  subnet_ids          = module.network.private_subnet_ids
  engine_version      = "16.1"
  instance_class      = "db.t4g.medium"
  allocated_storage   = 100
  multi_az            = true
  backup_retention    = 7
  deletion_protection = true
  tags                = local.common_tags
}

module "compute" {
  source = "../../modules/compute"

  name             = "${local.env}-webapp"
  vpc_id           = module.network.vpc_id
  subnet_ids       = module.network.private_subnet_ids
  instance_type    = "t3.medium"
  min_size         = 3
  max_size         = 12
  desired_capacity = 3

  database_endpoint  = module.data.endpoint
  database_secret_arn = module.data.password_secret_arn

  tags = local.common_tags
}

module "edge" {
  source = "../../modules/edge"

  name                  = "${local.env}-webapp"
  vpc_id                = module.network.vpc_id
  public_subnet_ids     = module.network.public_subnet_ids
  target_group_arn      = module.compute.target_group_arn
  domain                = "app.example.com"
  acm_certificate_arn   = var.acm_certificate_arn

  tags = local.common_tags
}

output "app_url"      { value = "https://${module.edge.cloudfront_domain}" }
output "alb_dns_name" { value = module.edge.alb_dns_name }
output "vpc_id"       { value = module.network.vpc_id }
```

What this is *not*. A 2000-line monolith with every resource inline. The orchestration is ~80 lines. All the real complexity — the VPC's subnet calculations, the ASG's launch template, the RDS encryption, the ALB target group health checks — lives in the modules where it can be tested and reused for staging and dev.

**`terraform.tfvars`** for prod is small — just the values that differ between environments:

```hcl
region              = "us-east-1"
vpc_cidr            = "10.30.0.0/16"
azs                 = ["us-east-1a", "us-east-1b", "us-east-1c"]
acm_certificate_arn = "arn:aws:acm:us-east-1:123456789012:certificate/abcd1234"
```

Staging gets a different `tfvars` with `vpc_cidr = "10.20.0.0/16"`, smaller instance sizes, `multi_az = false`. The root `main.tf` doesn't change between environments — only the inputs.

This is the payoff of every pattern in the series. **The variation is at the boundary, not in the code.**


## Organizing state at scale — blast radius

When the infrastructure grows past a single Terraform root, the question becomes: where do you split state?

**The blast-radius rule.** Group resources whose simultaneous failure is acceptable; split apart anything where co-failure is catastrophic. Concretely:

- **One state file per environment** — yes, always. Dev plans must not see prod state.
- **One state file per major component** — when components have independent change cadences. Network changes monthly; the app changes daily. Splitting them means an app deploy doesn't replan the network.
- **One state file per team boundary** — when different teams own different layers. The platform team owns network and IAM; the app team owns compute and data. Each team has its own state and its own pipeline.

```
   live/
   ├── prod/
   │   ├── network/      <- platform team, state key "live/prod/network"
   │   ├── iam/          <- platform team, state key "live/prod/iam"
   │   ├── data/         <- data team, state key "live/prod/data"
   │   ├── compute/      <- app team, state key "live/prod/compute"
   │   └── edge/         <- app team, state key "live/prod/edge"
   ├── staging/  ...
   └── dev/      ...
```

Cross-state references go through `terraform_remote_state` data sources or first-class data sources with stable tags (notebook 07).

**The trade-off.** More state files mean more pipelines, more `terraform_remote_state` lookups, more orchestration. Don't split prematurely. The rule of thumb: a single state file with more than 200 resources is starting to be too coarse; over 500 is definitely too coarse. Plan times alone become annoying.

**The opposite mistake.** Splitting too aggressively — every micro-component in its own state — creates a coordination nightmare. A single change requires applying five states in the right order. Find the natural seams (team boundaries, change-cadence boundaries) and split there.


## Drift detection in production

The state file is Terraform's idea of the world. Cloud reality may diverge — someone makes a console change, an external tool modifies a tag, an autoscaler updates a count Terraform thought was fixed.

**The detection pipeline:**

```
   nightly cron (or hourly)
        |
        v
   for each state file:
        terraform plan -refresh-only -detailed-exitcode
        |
        v
   exit code 0  -> no drift, do nothing
   exit code 1  -> error (auth, network), page oncall
   exit code 2  -> drift detected, post to Slack with diff
```

`-refresh-only` makes the plan ignore code changes and report only the cloud-state-vs-state-file diff. `-detailed-exitcode` returns 2 when changes exist, which CI can branch on.

**What to do when drift is detected:**

- **Investigate first, don't apply.** Drift often means someone fixed something out of band. Reverting it via Terraform may be the wrong move.
- **Bring the drift into code.** If the console change was correct, update the HCL to match and apply (which becomes a no-op after refresh).
- **Revert if the drift is unauthorized.** If someone changed prod by hand without a PR, revert and discuss what happened.

**Production tools:**

- **HCP Terraform** and **Spacelift** both ship drift detection. They run periodic refreshes and surface drift in the UI.
- **AWS Config** rules can flag specific kinds of drift (open security groups, untagged resources).
- **Custom CloudTrail filters** detect API calls outside the Terraform pipeline's IAM role — "if anyone but Terraform modifies prod resources, alert."

The honest framing: drift in production is *information*. Treat it as a leading indicator that someone bypassed the pipeline, or that the IAM boundary isn't tight enough.


## Cost control — tagging and estimation

Terraform makes it easy to create infrastructure quickly. That's the feature and the bill. Three patterns control costs.

**1. Tagging discipline.** Every resource gets a consistent set of tags. The canonical minimum:

```hcl
locals {
  required_tags = {
    Environment = "prod"
    Service     = "webapp"
    Owner       = "platform-team@example.com"
    CostCenter  = "engineering"
    ManagedBy   = "terraform"
  }
}
```

Apply these via a default-tags provider config so you don't have to remember:

```hcl
provider "aws" {
  region = var.region
  default_tags {
    tags = local.required_tags
  }
}
```

Every resource the provider creates picks up these tags automatically. Resource-specific tags merge on top. **AWS Cost Explorer can then break down spend by tag** — by service, by team, by environment. Without consistent tagging, your cost report is one undifferentiated lump.

**Enforce tagging with policies.** AWS Organizations Service Control Policies, OPA, Sentinel — all let you reject resource creation that doesn't carry the required tags. The earlier the rejection (at apply, not at billing review), the cheaper the lesson.

**2. Infracost in PR reviews.** [Infracost](https://infracost.io/) estimates the dollar impact of a Terraform plan and posts it as a PR comment:

```
This PR will add $341/month of AWS spend:
  + aws_db_instance.replica          $180/month
  + aws_instance.web                 $122/month
  + aws_ebs_volume.data              $39/month
```

A reviewer who sees "$340/month" on a typo-fix PR knows something's wrong before approving. Infracost is open-source and integrates with every major CI; some commercial tools (Spacelift, env0, HCP Terraform) include similar functionality.

**3. The money-burning patterns to watch for:**

- **NAT gateways per AZ.** $32/month each, plus data-transfer charges. Three AZs in prod = $100/month before any traffic. Fine for prod; overkill for dev. Use `single_nat_gateway` in non-prod environments.
- **Idle RDS instances and read replicas.** Provisioned even when no traffic flows. Use serverless v2 or stop instances in non-prod.
- **EBS snapshots and stale AMIs.** They accumulate forever. Set lifecycle policies.
- **CloudWatch logs without retention.** Default retention is "never expire." Set retention explicitly: `retention_in_days = 30`.
- **Forgotten dev environments.** `terraform destroy` at the end of every working day in dev; CI to alert on long-lived "test" stacks.

The line that separates productive Terraform from expensive Terraform is one of *visibility* — knowing what you're spending before the month closes.


## The debugging workflow

When Terraform misbehaves, the workflow:

### Step 1 — Read the error carefully

Terraform's errors are usually specific. "Provider produced inconsistent final plan" means the provider's logic returned different results between plan and apply. "Cycle in graph" means circular references. The error text tells you what.

### Step 2 — `TF_LOG=DEBUG`

For deeper investigation, raise the log level:

```bash
$ TF_LOG=DEBUG terraform apply 2> debug.log
```

Levels: `TRACE`, `DEBUG`, `INFO`, `WARN`, `ERROR`. `TRACE` is the firehose — provider API calls, state reads, every operation. Use `DEBUG` first; escalate to `TRACE` if needed.

`TF_LOG_PATH=path/to/log` separates Terraform's logs from the user-facing output. Useful when you want to capture logs without losing the readable summary.

### Step 3 — `terraform state list` and `terraform state show`

When the question is "what does Terraform think exists?" — `state list` for addresses, `state show <address>` for details. The single best diagnostic tool for "I have no idea what's happening."

### Step 4 — `terraform plan` with the smallest possible change

If a single PR's plan looks scary, *back the change down to the minimum*. Comment out everything except the one resource you care about; plan; verify; uncomment incrementally. This narrows what's actually different.

### Step 5 — Provider-specific debugging

For AWS, AWS API errors usually have specific codes (`InvalidParameterValue`, `LimitExceeded`, `Unauthorized`). The Terraform provider passes them through. Search the code; the AWS docs explain what triggers each.

`AWS_SDK_LOAD_CONFIG=1 AWS_PROFILE=foo terraform plan` to confirm credentials. `aws sts get-caller-identity` to verify what role is being assumed.

### Common error patterns

- **`Error: Provider produced inconsistent final plan`** — usually a provider bug, sometimes user error (refrencing a value that depends on `apply`-time computation in an unexpected place). Try `-target` to isolate; report to the provider repo if it's the provider.
- **`Error: state is locked`** — covered in notebook 03. `force-unlock` only after confirming no live run.
- **`Error: BucketAlreadyExists`** — S3 bucket names are globally unique. Add a random suffix.
- **`Error: Cycle in graph`** — circular reference. `terraform graph` to visualize. Usually fixed by removing a `depends_on` or restructuring how a value flows.
- **`Plan: 0 to add, 0 to change, 0 to destroy. Outputs change: ...`** — outputs depend on resource attributes that haven't been computed. Often harmless; sometimes a sign of `null` propagation.


## When Terraform is the wrong tool

After 8 notebooks of Terraform advocacy, the honest counterweight: **Terraform isn't always the right tool**.

**Cases where Terraform fits poorly:**

- **Highly imperative, ephemeral workflows.** Spinning up a one-off batch job, running it, tearing it down — the cloud's native job APIs or a tool like Nomad are better fits than declarative IaC.
- **Application configuration.** Database schema migrations, application secrets that rotate frequently, feature flag values. These belong in app-deployment tooling (Flyway, Liquibase, app-config services), not Terraform.
- **Per-request infrastructure.** Spawning a Lambda for each user request — that's the Lambda invocation model, not Terraform's. Terraform manages the Lambda's existence; the runtime handles invocation.
- **Continuous reconciliation.** If you need the controller to *constantly* watch for drift and reconcile within seconds (Kubernetes operators, Crossplane), Terraform's plan-and-apply model is the wrong shape. Crossplane or operators are built for this; Terraform isn't.
- **State the cloud already manages well.** CloudFormation StackSets, AWS Service Catalog, and similar AWS-native tools have access to data Terraform doesn't, and integrate more deeply with AWS-only operations like rollback.

**When to combine tools:**

- **Terraform for static infrastructure + Helm for Kubernetes workloads.** Terraform creates the EKS cluster; Helm deploys the apps onto it. The boundary is clear: Terraform owns what Kubernetes can't, Helm owns what's inside Kubernetes.
- **Terraform for cloud resources + Ansible for on-host configuration.** Notebook 07's provisioner alternative.
- **Terraform + ArgoCD.** Same EKS-Helm pattern, with GitOps for app deployment.

The honest framing: Terraform is the best tool for **declarative, change-batched, cloud-resource lifecycle**. When your problem has that shape, use it. When it doesn't, use the right shape's tool.


## The throughline

Eight notebooks in one paragraph.

**Terraform's contribution to the field isn't HCL or the provider ecosystem — it's the plan-apply workflow.** Every change is previewed before it happens. Every reviewer sees what every operator will run. The diff between intention and reality is the work product. State management exists to make that diff possible; modules exist to make the intention readable; environments exist to keep the changes scoped; refactoring features exist so the configuration can evolve without surprise.

Three habits, drawn from across the notebooks, separate engineers who use Terraform well from those who fight it:

- **Read the plan.** Every single time. The `-/+` symbol is doing more to prevent outages than any other single feature.
- **Treat state as load-bearing.** Remote backend, locking, encryption, limited access, never in git. State is the same risk category as the database itself.
- **Make the variation live at the boundary.** Inputs to modules differ between environments; the module's internals don't. The further you can push variation out toward `tfvars` and away from `if` ladders inside modules, the cleaner the system stays.

That's the series. Write declarative, review the plan, manage state carefully, and remember that infrastructure is something a team agrees on — not something one person clicks into existence and hopes the rest can follow.
